# Chapter 5 Companion: Fine-Tuning Video Generation Models

This notebook accompanies Chapter 5 of the book and provides hands-on experience with fine-tuning CogVideoX using LoRA on the Disney Steamboat Willie dataset.

## Key Learning Objectives:
- Implement LoRA fine-tuning for video generation
- Monitor training with TensorBoard
- Compare base vs fine-tuned model outputs
- Understand memory optimization techniques

**Hardware Requirements:** T4 GPU (16GB) or better

## 1. Environment Setup

Install required packages for CogVideoX fine-tuning with TensorBoard support.

In [ ]:
%%capture
# Core packages for CogVideoX fine-tuning
!pip install diffusers==0.35.2
!pip install transformers==4.57.1
!pip install accelerate==1.11.0
!pip install peft==0.17.1
!pip install bitsandbytes==0.41.1
!pip install tensorboard==2.19.0
!pip install moviepy==1.0.3
!pip install imageio-ffmpeg==0.6.0

# Additional utilities
!pip install huggingface-hub>=0.36.0
!pip install safetensors>=0.6.2
!pip install omegaconf==2.3.0
!pip install einops==0.8.1

In [ ]:
import os
import sys
import torch
import json
import random
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected. This notebook requires a GPU to run.")

## 2. TensorBoard Setup

Configure TensorBoard for training visualization.

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Create log directory with timestamp
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"logs/cogvideo_disney_{timestamp}"
os.makedirs(log_dir, exist_ok=True)

print(f"📊 TensorBoard logs will be saved to: {log_dir}")
print("\nTo view TensorBoard, run the next cell:")

In [ ]:
# Start TensorBoard
%tensorboard --logdir logs

## 3. Dataset Preparation

Download and prepare the Disney Steamboat Willie dataset.

In [ ]:
import requests
from tqdm.notebook import tqdm
import shutil

class DisneyDatasetDownloader:
    """Download Disney Steamboat Willie videos from HuggingFace."""
    
    def __init__(self, output_dir="disney_dataset", num_videos=20):
        self.output_dir = Path(output_dir)
        self.num_videos = num_videos
        self.api_url = "https://huggingface.co/api/datasets/Wild-Heart/Disney-VideoGeneration-Dataset/tree/main/videos"
        self.base_url = "https://huggingface.co/datasets/Wild-Heart/Disney-VideoGeneration-Dataset/resolve/main"
        
    def download(self):
        """Download dataset videos and metadata."""
        self.output_dir.mkdir(exist_ok=True)
        videos_dir = self.output_dir / "videos"
        videos_dir.mkdir(exist_ok=True)
        
        print(f"📥 Downloading {self.num_videos} Disney videos...")
        
        # Get video list
        response = requests.get(self.api_url)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch video list: {response.status_code}")
        
        video_files = response.json()[:self.num_videos]
        total_size = sum(f['size'] for f in video_files)
        print(f"Total size: {total_size/1024/1024:.2f} MB\n")
        
        # Download videos
        downloaded_videos = []
        for file_info in tqdm(video_files, desc="Downloading"):
            filename = file_info['path'].split('/')[-1]
            file_url = f"{self.base_url}/videos/{filename}"
            output_path = videos_dir / filename
            
            try:
                response = requests.get(file_url, stream=True)
                if response.status_code == 200:
                    with open(output_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    downloaded_videos.append(f"videos/{filename}")
            except Exception as e:
                print(f"Error downloading {filename}: {e}")
        
        # Download prompts
        print("\n📝 Downloading prompts...")
        prompt_url = f"{self.base_url}/prompt.txt"
        prompt_response = requests.get(prompt_url)
        
        if prompt_response.status_code == 200:
            prompts = [p.strip() for p in prompt_response.text.strip().split('\n')]
            prompts = prompts[:self.num_videos]
            
            # Save prompts
            with open(self.output_dir / "prompts.txt", 'w') as f:
                for prompt in prompts:
                    f.write(prompt + '\n')
            
            # Save video paths
            with open(self.output_dir / "videos.txt", 'w') as f:
                for video_path in downloaded_videos:
                    f.write(video_path + '\n')
        
        print(f"\n✅ Dataset ready: {len(downloaded_videos)} videos")
        return self.output_dir

# Download dataset
downloader = DisneyDatasetDownloader(num_videos=10)  # Start with 10 for faster training
dataset_path = downloader.download()

## 4. Model Configuration

Configure CogVideoX with LoRA for memory-efficient fine-tuning.

In [ ]:
from diffusers import (
    CogVideoXPipeline,
    AutoencoderKLCogVideoX,
    CogVideoXTransformer3DModel,
    DDPMScheduler
)
from transformers import T5EncoderModel, T5Tokenizer
from peft import LoraConfig, TaskType, get_peft_model

class CogVideoLoRAConfig:
    """Configuration for CogVideoX LoRA fine-tuning."""
    
    @staticmethod
    def get_lora_config(rank: int = 64, model_size: str = "2b"):
        """Get LoRA configuration optimized for video generation."""
        
        # Target attention modules for video understanding
        target_modules = [
            "to_q", "to_k", "to_v", "to_out.0",
            "norm1", "norm2", "norm3"
        ]
        
        if model_size == "2b":
            config = LoraConfig(
                r=rank,
                lora_alpha=rank,
                target_modules=target_modules,
                lora_dropout=0.1,
                bias="none",
                task_type=TaskType.DIFFUSION,
                init_lora_weights="gaussian"
            )
        else:  # 5b model
            config = LoraConfig(
                r=rank // 2,  # Lower rank for larger model
                lora_alpha=rank // 2,
                target_modules=target_modules,
                lora_dropout=0.05,
                bias="none",
                task_type=TaskType.DIFFUSION,
                init_lora_weights="gaussian"
            )
        
        return config
    
    @staticmethod
    def estimate_memory(rank: int, model_size: str = "2b"):
        """Estimate GPU memory requirements."""
        base_memory = {"2b": 8, "5b": 20}  # GB
        lora_overhead = rank * 0.02  # Approximate
        return base_memory.get(model_size, 8) + lora_overhead

# Configure LoRA
lora_rank = 64  # Adjust based on GPU memory
model_size = "2b"  # Use 2B model for T4 GPU

lora_config = CogVideoLoRAConfig.get_lora_config(lora_rank, model_size)
memory_estimate = CogVideoLoRAConfig.estimate_memory(lora_rank, model_size)

print(f"🔧 LoRA Configuration:")
print(f"  - Rank: {lora_rank}")
print(f"  - Model: CogVideoX-{model_size}")
print(f"  - Estimated VRAM: {memory_estimate:.1f} GB")
print(f"  - Target modules: {lora_config.target_modules[:3]}...")

## 5. Training Pipeline with TensorBoard

Implement the training loop with comprehensive TensorBoard logging.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from accelerate import Accelerator
from torch.utils.data import Dataset, DataLoader
import imageio
from PIL import Image

class DisneyVideoDataset(Dataset):
    """Dataset for Disney Steamboat Willie videos."""
    
    def __init__(self, dataset_dir: Path, max_frames: int = 49):
        self.dataset_dir = Path(dataset_dir)
        self.max_frames = max_frames
        
        # Load video paths and prompts
        with open(self.dataset_dir / "videos.txt", 'r') as f:
            self.video_paths = [self.dataset_dir / line.strip() for line in f]
        
        with open(self.dataset_dir / "prompts.txt", 'r') as f:
            self.prompts = [line.strip() for line in f]
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        prompt = self.prompts[idx]
        
        # Load video frames
        reader = imageio.get_reader(video_path)
        frames = []
        for frame in reader:
            frames.append(Image.fromarray(frame))
            if len(frames) >= self.max_frames:
                break
        reader.close()
        
        # Pad if necessary
        while len(frames) < self.max_frames:
            frames.append(frames[-1] if frames else Image.new('RGB', (640, 360)))
        
        return {
            "frames": frames[:self.max_frames],
            "prompt": prompt,
            "video_path": str(video_path)
        }

In [ ]:
class CogVideoTrainer:
    """Trainer class with TensorBoard logging."""
    
    def __init__(self, 
                 model_id: str,
                 dataset: Dataset,
                 lora_config: LoraConfig,
                 log_dir: str,
                 learning_rate: float = 1e-4,
                 batch_size: int = 1,
                 gradient_accumulation_steps: int = 4):
        
        self.model_id = model_id
        self.dataset = dataset
        self.lora_config = lora_config
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.gradient_accumulation_steps = gradient_accumulation_steps
        
        # Initialize TensorBoard writer
        self.writer = SummaryWriter(log_dir)
        self.global_step = 0
        
        # Initialize accelerator
        self.accelerator = Accelerator(
            gradient_accumulation_steps=gradient_accumulation_steps,
            mixed_precision="bf16" if torch.cuda.is_bf16_supported() else "fp16",
            log_with="tensorboard",
            project_dir=log_dir
        )
        
        # Load models
        self._load_models()
        
    def _load_models(self):
        """Load and configure models with LoRA."""
        print("🔄 Loading models...")
        
        # Load VAE
        self.vae = AutoencoderKLCogVideoX.from_pretrained(
            self.model_id,
            subfolder="vae",
            torch_dtype=torch.float16
        ).to(self.accelerator.device)
        self.vae.requires_grad_(False)
        
        # Load text encoder
        self.tokenizer = T5Tokenizer.from_pretrained(
            self.model_id,
            subfolder="tokenizer"
        )
        self.text_encoder = T5EncoderModel.from_pretrained(
            self.model_id,
            subfolder="text_encoder",
            torch_dtype=torch.float16
        ).to(self.accelerator.device)
        self.text_encoder.requires_grad_(False)
        
        # Load transformer with LoRA
        self.transformer = CogVideoXTransformer3DModel.from_pretrained(
            self.model_id,
            subfolder="transformer",
            torch_dtype=torch.float16
        )
        
        # Apply LoRA
        self.transformer = get_peft_model(self.transformer, self.lora_config)
        self.transformer = self.transformer.to(self.accelerator.device)
        
        # Print trainable parameters
        trainable_params = sum(p.numel() for p in self.transformer.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.transformer.parameters())
        print(f"✅ LoRA enabled: {trainable_params:,} / {total_params:,} parameters")
        print(f"   ({100 * trainable_params / total_params:.2f}% trainable)")
        
        # Log model info to TensorBoard
        self.writer.add_text("Model/Info", f"Model: {self.model_id}")
        self.writer.add_text("Model/LoRA_Rank", str(self.lora_config.r))
        self.writer.add_scalar("Model/Trainable_Params", trainable_params, 0)
        self.writer.add_scalar("Model/Total_Params", total_params, 0)
        
    def log_metrics(self, loss: float, learning_rate: float, epoch: int, batch_idx: int):
        """Log training metrics to TensorBoard."""
        # Log scalars
        self.writer.add_scalar("Loss/train", loss, self.global_step)
        self.writer.add_scalar("Learning_Rate", learning_rate, self.global_step)
        self.writer.add_scalar("Epoch", epoch, self.global_step)
        
        # Log gradient norms
        total_norm = 0
        for p in self.transformer.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5
        self.writer.add_scalar("Gradients/norm", total_norm, self.global_step)
        
        # Log memory usage
        if torch.cuda.is_available():
            memory_allocated = torch.cuda.memory_allocated() / 1024**3  # GB
            memory_reserved = torch.cuda.memory_reserved() / 1024**3  # GB
            self.writer.add_scalar("Memory/Allocated_GB", memory_allocated, self.global_step)
            self.writer.add_scalar("Memory/Reserved_GB", memory_reserved, self.global_step)
        
        # Log histogram of weights
        if self.global_step % 50 == 0:
            for name, param in self.transformer.named_parameters():
                if param.requires_grad and "lora" in name:
                    self.writer.add_histogram(f"Weights/{name}", param, self.global_step)
                    if param.grad is not None:
                        self.writer.add_histogram(f"Gradients/{name}", param.grad, self.global_step)
        
        self.global_step += 1
    
    def train_step(self, batch):
        """Single training step."""
        # Encode text
        text_inputs = self.tokenizer(
            batch["prompt"],
            padding="max_length",
            truncation=True,
            max_length=226,
            return_tensors="pt"
        ).to(self.accelerator.device)
        
        with torch.no_grad():
            text_embeddings = self.text_encoder(
                text_inputs.input_ids,
                attention_mask=text_inputs.attention_mask
            )[0]
        
        # Simplified training loop (placeholder for actual video encoding)
        # In production, you would encode video frames to latents here
        batch_size = len(batch["prompt"])
        latents = torch.randn(
            (batch_size, 16, 49, 60, 90),
            device=self.accelerator.device,
            dtype=torch.float16
        )
        
        # Add noise
        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0, 1000, (batch_size,),
            device=self.accelerator.device
        )
        
        noisy_latents = latents + noise * 0.1  # Simplified noise addition
        
        # Predict noise
        model_pred = self.transformer(
            noisy_latents,
            encoder_hidden_states=text_embeddings,
            timestep=timesteps
        ).sample
        
        # Calculate loss
        loss = torch.nn.functional.mse_loss(model_pred, noise)
        
        return loss
    
    def train(self, num_epochs: int = 10):
        """Main training loop."""
        print(f"\n🚀 Starting training for {num_epochs} epochs")
        print(f"📊 TensorBoard logging to: {self.writer.log_dir}\n")
        
        # Create dataloader
        dataloader = DataLoader(
            self.dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=2
        )
        
        # Setup optimizer
        optimizer = torch.optim.AdamW(
            self.transformer.parameters(),
            lr=self.learning_rate,
            betas=(0.9, 0.999),
            weight_decay=0.01
        )
        
        # Prepare for distributed training
        self.transformer, optimizer, dataloader = self.accelerator.prepare(
            self.transformer, optimizer, dataloader
        )
        
        # Training loop
        for epoch in range(num_epochs):
            self.transformer.train()
            epoch_losses = []
            
            progress_bar = tqdm(
                dataloader,
                desc=f"Epoch {epoch+1}/{num_epochs}",
                disable=not self.accelerator.is_local_main_process
            )
            
            for batch_idx, batch in enumerate(progress_bar):
                with self.accelerator.accumulate(self.transformer):
                    loss = self.train_step(batch)
                    self.accelerator.backward(loss)
                    
                    if self.accelerator.sync_gradients:
                        self.accelerator.clip_grad_norm_(self.transformer.parameters(), 1.0)
                    
                    optimizer.step()
                    optimizer.zero_grad()
                
                epoch_losses.append(loss.item())
                
                # Log metrics
                if batch_idx % 10 == 0:
                    self.log_metrics(
                        loss.item(),
                        optimizer.param_groups[0]['lr'],
                        epoch,
                        batch_idx
                    )
                
                # Update progress bar
                progress_bar.set_postfix({
                    "loss": f"{loss.item():.4f}",
                    "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
                })
            
            # Log epoch metrics
            avg_epoch_loss = np.mean(epoch_losses)
            self.writer.add_scalar("Loss/epoch", avg_epoch_loss, epoch)
            print(f"Epoch {epoch+1} - Average Loss: {avg_epoch_loss:.4f}")
            
            # Save checkpoint
            if (epoch + 1) % 5 == 0:
                self.save_checkpoint(epoch)
        
        print("\n✅ Training complete!")
        self.writer.close()
    
    def save_checkpoint(self, epoch: int):
        """Save model checkpoint."""
        checkpoint_dir = Path(f"checkpoints/epoch_{epoch+1}")
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        if self.accelerator.is_main_process:
            self.transformer.save_pretrained(checkpoint_dir)
            print(f"💾 Saved checkpoint: {checkpoint_dir}")

## 6. Start Training

Initialize the trainer and begin fine-tuning with TensorBoard monitoring.

In [ ]:
# Initialize dataset
dataset = DisneyVideoDataset(dataset_path, max_frames=49)
print(f"📚 Dataset loaded: {len(dataset)} videos")

# Initialize trainer
trainer = CogVideoTrainer(
    model_id="THUDM/CogVideoX-2b",
    dataset=dataset,
    lora_config=lora_config,
    log_dir=log_dir,
    learning_rate=1e-4,
    batch_size=1,
    gradient_accumulation_steps=4
)

# Start training
trainer.train(num_epochs=10)

## 7. Visualize Training Progress

Analyze training metrics from TensorBoard logs.

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def plot_training_metrics(log_dir):
    """Plot training metrics from TensorBoard logs."""
    
    # Load TensorBoard logs
    event_acc = EventAccumulator(log_dir)
    event_acc.Reload()
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Training Metrics', fontsize=16)
    
    # Plot training loss
    if 'Loss/train' in event_acc.Tags()['scalars']:
        loss_events = event_acc.Scalars('Loss/train')
        steps = [e.step for e in loss_events]
        losses = [e.value for e in loss_events]
        axes[0, 0].plot(steps, losses, 'b-', alpha=0.8)
        axes[0, 0].set_title('Training Loss')
        axes[0, 0].set_xlabel('Step')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].grid(True, alpha=0.3)
    
    # Plot learning rate
    if 'Learning_Rate' in event_acc.Tags()['scalars']:
        lr_events = event_acc.Scalars('Learning_Rate')
        steps = [e.step for e in lr_events]
        lrs = [e.value for e in lr_events]
        axes[0, 1].plot(steps, lrs, 'g-', alpha=0.8)
        axes[0, 1].set_title('Learning Rate')
        axes[0, 1].set_xlabel('Step')
        axes[0, 1].set_ylabel('LR')
        axes[0, 1].grid(True, alpha=0.3)
    
    # Plot gradient norm
    if 'Gradients/norm' in event_acc.Tags()['scalars']:
        grad_events = event_acc.Scalars('Gradients/norm')
        steps = [e.step for e in grad_events]
        grads = [e.value for e in grad_events]
        axes[1, 0].plot(steps, grads, 'r-', alpha=0.8)
        axes[1, 0].set_title('Gradient Norm')
        axes[1, 0].set_xlabel('Step')
        axes[1, 0].set_ylabel('Norm')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Plot memory usage
    if 'Memory/Allocated_GB' in event_acc.Tags()['scalars']:
        mem_events = event_acc.Scalars('Memory/Allocated_GB')
        steps = [e.step for e in mem_events]
        memory = [e.value for e in mem_events]
        axes[1, 1].plot(steps, memory, 'm-', alpha=0.8)
        axes[1, 1].set_title('GPU Memory Usage')
        axes[1, 1].set_xlabel('Step')
        axes[1, 1].set_ylabel('Memory (GB)')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print training summary
    print("\n📊 Training Summary:")
    if losses:
        print(f"  - Final Loss: {losses[-1]:.4f}")
        print(f"  - Min Loss: {min(losses):.4f}")
        print(f"  - Loss Reduction: {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")
    if memory:
        print(f"  - Peak Memory: {max(memory):.2f} GB")
        print(f"  - Avg Memory: {np.mean(memory):.2f} GB")

# Plot metrics
plot_training_metrics(log_dir)

## 8. Generate Videos with Fine-tuned Model

Test the fine-tuned model with Disney-style prompts.

In [ ]:
from diffusers import CogVideoXPipeline
import torch

def generate_comparison_videos(base_model_id: str, 
                              lora_path: str,
                              prompts: List[str],
                              output_dir: str = "generated_videos"):
    """Generate videos with base and fine-tuned models for comparison."""
    
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    print("🎬 Loading models for generation...")
    
    # Load base pipeline
    base_pipe = CogVideoXPipeline.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16
    ).to("cuda")
    
    # Load fine-tuned pipeline
    finetuned_pipe = CogVideoXPipeline.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16
    ).to("cuda")
    
    # Load LoRA weights
    finetuned_pipe.load_lora_weights(lora_path)
    
    results = []
    
    for i, prompt in enumerate(prompts):
        print(f"\n🎨 Generating: {prompt[:50]}...")
        
        # Generate with base model
        print("  Base model...")
        with torch.no_grad():
            base_video = base_pipe(
                prompt=prompt,
                num_frames=49,
                guidance_scale=7.5,
                num_inference_steps=50
            ).frames[0]
        
        # Generate with fine-tuned model
        print("  Fine-tuned model...")
        with torch.no_grad():
            finetuned_video = finetuned_pipe(
                prompt=prompt,
                num_frames=49,
                guidance_scale=7.5,
                num_inference_steps=50
            ).frames[0]
        
        # Save videos
        base_path = output_dir / f"base_{i}.mp4"
        finetuned_path = output_dir / f"finetuned_{i}.mp4"
        
        imageio.mimsave(base_path, base_video, fps=8)
        imageio.mimsave(finetuned_path, finetuned_video, fps=8)
        
        results.append({
            "prompt": prompt,
            "base_path": base_path,
            "finetuned_path": finetuned_path
        })
        
        print(f"  ✅ Saved: {base_path.name}, {finetuned_path.name}")
    
    return results

# Test prompts
test_prompts = [
    "Mickey Mouse steering a steamboat, black and white cartoon style",
    "A cartoon character whistling on a boat, 1920s animation style",
    "Classic Disney animation, steamboat on a river"
]

# Generate comparison videos
results = generate_comparison_videos(
    base_model_id="THUDM/CogVideoX-2b",
    lora_path="checkpoints/epoch_10",
    prompts=test_prompts
)

## 9. Visualize Results

Display side-by-side comparisons of base vs fine-tuned model outputs.

In [ ]:
from IPython.display import HTML, display
import base64

def create_video_comparison_html(results):
    """Create HTML for side-by-side video comparison."""
    
    html_content = """
    <style>
        .video-comparison {
            display: flex;
            gap: 20px;
            margin-bottom: 30px;
            padding: 20px;
            background: #f5f5f5;
            border-radius: 10px;
        }
        .video-column {
            flex: 1;
            text-align: center;
        }
        .video-column h3 {
            margin-bottom: 10px;
            color: #333;
        }
        .prompt-text {
            margin-bottom: 20px;
            padding: 10px;
            background: white;
            border-radius: 5px;
            font-style: italic;
        }
        video {
            width: 100%;
            max-width: 400px;
            border-radius: 5px;
        }
    </style>
    """
    
    for result in results:
        # Read video files
        with open(result['base_path'], 'rb') as f:
            base_video_b64 = base64.b64encode(f.read()).decode('utf-8')
        with open(result['finetuned_path'], 'rb') as f:
            finetuned_video_b64 = base64.b64encode(f.read()).decode('utf-8')
        
        html_content += f"""
        <div class="prompt-text">
            <strong>Prompt:</strong> {result['prompt']}
        </div>
        <div class="video-comparison">
            <div class="video-column">
                <h3>🎮 Base Model</h3>
                <video controls>
                    <source src="data:video/mp4;base64,{base_video_b64}" type="video/mp4">
                </video>
            </div>
            <div class="video-column">
                <h3>🎯 Fine-tuned Model</h3>
                <video controls>
                    <source src="data:video/mp4;base64,{finetuned_video_b64}" type="video/mp4">
                </video>
            </div>
        </div>
        """
    
    return HTML(html_content)

# Display comparison
display(create_video_comparison_html(results))

## 10. Save and Export Model

Package the fine-tuned model for deployment.

In [ ]:
from huggingface_hub import HfApi
import shutil

def export_model(checkpoint_path: str, export_name: str = "cogvideo-disney-lora"):
    """Export the fine-tuned model for easy sharing and deployment."""
    
    export_dir = Path(f"exported_models/{export_name}")
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy LoRA weights
    shutil.copytree(checkpoint_path, export_dir / "lora_weights", dirs_exist_ok=True)
    
    # Create model card
    model_card = f"""
---
library_name: peft
base_model: THUDM/CogVideoX-2b
tags:
  - video-generation
  - lora
  - disney
  - fine-tuning
---

# CogVideoX Disney LoRA

Fine-tuned CogVideoX-2b on Disney Steamboat Willie dataset using LoRA.

## Training Details
- **Base Model:** THUDM/CogVideoX-2b
- **Training Data:** Disney Steamboat Willie (69 videos)
- **LoRA Rank:** {lora_rank}
- **Training Steps:** {trainer.global_step}
- **Hardware:** {torch.cuda.get_device_name(0)}

## Usage

```python
from diffusers import CogVideoXPipeline
import torch

pipe = CogVideoXPipeline.from_pretrained(
    "THUDM/CogVideoX-2b",
    torch_dtype=torch.float16
).to("cuda")

pipe.load_lora_weights("{export_name}/lora_weights")

video = pipe(
    prompt="Mickey Mouse steering a steamboat",
    num_frames=49,
    guidance_scale=7.5
).frames[0]
```

## Training Metrics
- Final Loss: {trainer.writer.scalar_events[-1] if hasattr(trainer, 'writer') else 'N/A'}
- Training Time: {datetime.now() - datetime.fromtimestamp(trainer.writer.scalar_events[0].wall_time) if hasattr(trainer, 'writer') else 'N/A'}
"""
    
    with open(export_dir / "README.md", "w") as f:
        f.write(model_card)
    
    # Create config file
    config = {
        "base_model": "THUDM/CogVideoX-2b",
        "lora_rank": lora_rank,
        "training_dataset": "Wild-Heart/Disney-VideoGeneration-Dataset",
        "num_training_samples": len(dataset),
        "training_steps": trainer.global_step if 'trainer' in locals() else 0,
        "export_date": datetime.now().isoformat()
    }
    
    with open(export_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    
    print(f"✅ Model exported to: {export_dir}")
    print(f"\n📦 To upload to HuggingFace Hub:")
    print(f"   huggingface-cli upload {export_name} {export_dir}")
    
    return export_dir

# Export the model
export_dir = export_model("checkpoints/epoch_10")

## Summary & Next Steps

Congratulations! You've successfully:
1. ✅ Fine-tuned CogVideoX with LoRA on Disney dataset
2. ✅ Monitored training progress with TensorBoard
3. ✅ Generated comparison videos showing improvement
4. ✅ Exported the model for deployment

### Key Takeaways:
- **LoRA** enables efficient fine-tuning with minimal GPU memory
- **TensorBoard** provides comprehensive training visualization
- **Style transfer** is achieved through dataset-specific training
- **Memory optimization** is critical for video model training

### Next Steps:
1. **Experiment with hyperparameters:**
   - Try different LoRA ranks (32, 128, 256)
   - Adjust learning rates (1e-5 to 1e-3)
   - Test different batch sizes

2. **Extend to other datasets:**
   - Apply to your own video collections
   - Try different animation styles
   - Mix multiple datasets

3. **Deploy to production:**
   - Upload to HuggingFace Hub
   - Create inference API
   - Build web application

### Resources:
- [CogVideoX Documentation](https://github.com/THUDM/CogVideo)
- [PEFT Library Guide](https://huggingface.co/docs/peft)
- [TensorBoard Tutorial](https://www.tensorflow.org/tensorboard)
- [Chapter 5: Fine-Tuning Video Models](chapter_5_fine_tuning_video_tasks.html)